In [1]:
%pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.7/760.7 kB 33.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 49.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 65.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 194.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 158.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 808.8/808.8 kB 50.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 539.0/539.2 MB 320.6 MB/s eta 0:00:01^C:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 539.0/539.2 MB 320.6 MB/s eta 0:00:01Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import re
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
# 1. Load Parquet baseline file
parquet_path = "flying_drones_baseline_prod.parquet"
df = pd.read_parquet(parquet_path)
print(f"Loaded raw Parquet baseline: {len(df):,} records")

Loaded raw Parquet baseline: 219,997 records


In [5]:
# 2. Apply text sanitization (matches exact row alignment used for embeddings)
df['appln_title'] = df['appln_title'].fillna('').str.strip()
df['appln_abstract'] = df['appln_abstract'].fillna('').str.strip()
df['full_text'] = df['appln_title'] + ". " + df['appln_abstract']

short_mask = df['full_text'].str.strip().str.len() <= 10
df = df[~short_mask].copy().reset_index(drop=True)
print(f"Sanitized corpus ready: {len(df):,} records")

Sanitized corpus ready: 219,996 records


In [6]:
# 3. Load pre-computed vector embeddings
corpus_embeddings = np.load("flying_drones_embeddings.npy")
assert len(df) == corpus_embeddings.shape[0], f"Mismatch: {len(df)} DF rows vs {corpus_embeddings.shape[0]} embeddings"

In [7]:
# 4. Compute Centroid Similarity (Strategy A) if not already saved in parquet
if 'sim_score_centroid' not in df.columns:
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
    
    seed_candidates = df[
        df['cpc_codes'].str.contains('B64U', na=False) & 
        df['appln_title'].str.contains('unmanned aerial vehicle|quadcopter|multirotor', case=False, na=False)
    ].head(15)
    
    seed_embeddings = model.encode(seed_candidates['full_text'].tolist(), convert_to_numpy=True, normalize_embeddings=True)
    centroid_vector = np.mean(seed_embeddings, axis=0, keepdims=True)
    centroid_vector = centroid_vector / np.linalg.norm(centroid_vector)
    
    df['sim_score_centroid'] = cosine_similarity(corpus_embeddings, centroid_vector).flatten()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 1ST VALIDATION STRATEGY: CPC/IPC CLASSIFICATION MIGRATION AUDIT

In [8]:
# 5. Extract CPC 4-character subclasses and compute migration audit
def extract_cpc_subclasses(cpc_series):
    all_codes = cpc_series.dropna().astype(str).str.findall(r'\b[A-H]\d{2}[A-Z]\b')
    flattened = [code for sublist in all_codes for code in set(sublist)]
    return pd.Series(flattened)

In [9]:
# Filter dataset using Strategy A threshold (>= 0.65)
df_filtered = df[df['sim_score_centroid'] >= 0.65].copy()

In [10]:
# Calculate subclass distributions
baseline_counts = extract_cpc_subclasses(df['cpc_codes']).value_counts(normalize=True) * 100
filtered_counts = extract_cpc_subclasses(df_filtered['cpc_codes']).value_counts(normalize=True) * 100

In [11]:
# Build comparison table
migration_df = pd.DataFrame({
    'Baseline Share (%)': baseline_counts,
    'Filtered Share (%)': filtered_counts
}).fillna(0)

In [12]:
migration_df['Share Change (pp)'] = migration_df['Filtered Share (%)'] - migration_df['Baseline Share (%)']
migration_df = migration_df.sort_values(by='Filtered Share (%)', ascending=False)

In [13]:
print("\n=== CPC Subclass Migration Audit (Top 15 Filtered Subclasses) ===")
print(migration_df.head(15).to_string())


=== CPC Subclass Migration Audit (Top 15 Filtered Subclasses) ===
      Baseline Share (%)  Filtered Share (%)  Share Change (pp)
B64U           20.030169           25.933298           5.903130
B64C            8.228600           17.591828           9.363227
B64D            7.597770           13.640802           6.043032
G05D            8.171252           10.594907           2.423655
B64F            2.231367            4.645835           2.414468
G08G            2.311655            2.764215           0.452561
G06Q            2.390552            1.213100          -1.177452
G01S            2.611603            1.085405          -1.526198
H04N            1.954010            1.055359          -0.898651
G01C            2.860807            1.006535          -1.854272
B60P            0.310376            0.976489           0.666114
B60L            0.984652            0.938932          -0.045720
H04W            2.919546            0.830016          -2.089530
G06V            3.908368            0

Semantic filtering successfully cleans the candidate pool. 

-> The top-5 subclasses after filtering are all direct aeronautical and flight-control classifications: 
- B64U (UAV): increased from 20.03% to 25.93%
- B64C (Aircraft Structures, Rotors, Aerodynamics): more than doubled from 8.23% to 17.59%
- B64D (Aircraft Equipment, Payloads, Powerplants): nearly doubled.
- G05D (Autopilot, Flight Path & Attitude Control): rose from 8.17% to 10.59%.
- B64F (Ground Stations, Launching, Recovery Docks): more than doubled.

Together, these 5 core subclass classifications expanded from 46.3% of the baseline to 72.4% of the filtered dataset. 

-> Non-specific technology classes that typically create false positives show consistent relative decreases. 

## 2ND VALIDATION STRATEGY: BOUNDARY SENSITIVITY ANALYSIS 

In [ ]:
# 1. Define similarity score brackets around the 0.65 threshold
brackets = [
    ("Rejected Noise (0.55 <= sim < 0.60)", 0.55, 0.60),
    ("Boundary Zone Below Cutoff (0.60 <= sim < 0.65)", 0.60, 0.65),
    ("Retained Target Core (0.65 <= sim < 0.70)", 0.65, 0.70),
    ("High-Precision Strict Core (sim >= 0.70)", 0.70, 1.00)
]

print("=== BOUNDARY SENSITIVITY AUDIT ===")
for label, low, high in brackets:
    subset = df[(df['sim_score_centroid'] >= low) & (df['sim_score_centroid'] < high)]
    print(f"\n--- {label} | Count: {len(subset):,} patents ---")
    
    # Calculate percentage of patents explicitly containing UAV CPC code 'B64U'
    b64u_share = subset['cpc_codes'].str.contains('B64U', na=False).mean() * 100
    print(f"B64U (Dedicated UAV Class) Share: {b64u_share:.1f}%")
    
    # Display 3 random sample titles to inspect semantic content
    sample_titles = subset['appln_title'].sample(min(3, len(subset)), random_state=42).tolist()
    print("Sample Titles:")
    for idx, title in enumerate(sample_titles, 1):
        print(f"  {idx}. {title}")